# 10 — V2 test-set evaluation (one-shot, sealed)

**What this notebook does.** Applies the pipeline frozen in notebook 09
(`pipe_v2_stable_surface.joblib`) to the 61 sealed test transcripts — the **only** time
these documents are loaded in the entire project. The result is the final answer to RQ1:
does V2 stable + surface beat surface alone on held-out data?

**Non-negotiable rules:**
- This notebook loads `test/` exactly once. Do not re-run it after Block 8 has marked
  `test_set_used: true` in `frozen_spec.json` — the integrity check at Block 0 will
  stop you if you try.
- No feature selection, threshold tuning, or any decision that could benefit from
  seeing the test labels happens here. The frozen pipeline is loaded as-is and applied
  as-is.
- The extraction model is `qwen3.5:122b` (MODEL_A) — the same model the frozen pipeline
  was trained on.

## Block 0 — Configuration and integrity check

The `assert_consistent()` call is the last safety gate before the test set is opened:
it reloads `frozen_spec.json` and `pipe_v2_stable_surface.joblib` from disk, checks
that the pipeline's expected feature count matches the spec's `frozen_primary` list,
that the prompt SHA in the spec matches the prompt currently in force, and — critically
— that `test_set_used` is still `false`. If any of these fail, the notebook stops here.

In [19]:
import json, re, datetime, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score

from helpers import (
    MODEL_A as MODEL, BASE_URL, TEMPERATURE, MAX_TOKENS, SEED, EXPECTED_PROMPT_SHA,
    V2_PROMPT, PROMPT_SHA, assert_domain_blind,
    make_provider, boot_ci as _boot_ci, paired_bootstrap,
)

warnings.filterwarnings("ignore")

# Set to False after the first complete extraction run; re-running with True on an
# already-sealed notebook is stopped by the test_set_used check in the next cell.
RUN_LLM = True

DATA       = Path("fileDataset")
if not DATA.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA}. Put fileDataset/ (or fileDataset.zip) next to this notebook."
    )
OUT        = DATA / "outputs"
FROZEN_DIR = OUT / "frozen_pipeline_v2"
RAW_TEST   = OUT / "v2_test_raw.jsonl"
META_TEST  = OUT / "v2_test_run_metadata.json"

print(f"model      : {MODEL}")
print(f"frozen dir : {FROZEN_DIR}")
print(f"RAW_TEST   : {RAW_TEST}")
print(f"RUN_LLM    : {RUN_LLM}")


model      : qwen3.5:122b
frozen dir : fileDataset/outputs/frozen_pipeline_v2
RAW_TEST   : fileDataset/outputs/v2_test_raw.jsonl
RUN_LLM    : True


In [20]:
# reload spec + frozen artifact fresh and assert they agree -- the checks
# assert_consistent() used to bundle, now written out so the last gate before the
# sealed test set is opened is fully visible. The test_set_used check is the only
# automatic guard against re-evaluating after seeing the result.
spec_path = FROZEN_DIR / "frozen_spec.json"
spec  = json.loads(spec_path.read_text())
_art  = joblib.load(FROZEN_DIR / spec["model_artifact"])
_pipe = _art["pipeline"] if isinstance(_art, dict) and "pipeline" in _art else _art

FROZEN_PRIMARY_FEATURES = spec["feature_sets"]["frozen_primary"]
SURFACE                 = spec["feature_sets"]["SURFACE"]

_n_in = getattr(_pipe, "n_features_in_", None)
assert _n_in is None or _n_in == len(FROZEN_PRIMARY_FEATURES), (
    f"pipeline expects {_n_in} features, frozen_primary lists {len(FROZEN_PRIMARY_FEATURES)}")
if isinstance(_art, dict) and _art.get("features") is not None:
    assert list(_art["features"]) == list(FROZEN_PRIMARY_FEATURES), \
        "artifact feature list != spec frozen_primary"
assert PROMPT_SHA == EXPECTED_PROMPT_SHA == spec.get("prompt_sha256_16"), (
    f"prompt hash mismatch: running={PROMPT_SHA} expected={EXPECTED_PROMPT_SHA} "
    f"spec={spec.get('prompt_sha256_16')} -- do not evaluate under a drifted prompt")
assert spec.get("test_set_used") is False, (
    "frozen_spec.json already says test_set_used = True -- the test set has been "
    "evaluated; do not re-run this notebook")
assert_domain_blind()

print("frozen_spec consistent -- safe to proceed:")
print(f"  model_artifact        : {spec['model_artifact']}")
print(f"  test_set_used         : {spec['test_set_used']}   <- must be False")
print(f"  test_extraction_model : {spec['test_extraction_model']}")
print(f"  frozen_primary        : {len(FROZEN_PRIMARY_FEATURES)} cols")
print(f"  prompt sha256[:16]    : {spec['prompt_sha256_16']}")
_pb = spec["paired_bootstrap"]["frozen_primary_vs_surface"]
print(f"  CV: AUC {spec['frozen_primary_cv_auc']:.4f}  "
      f"paired dAUC {_pb['delta_auc']:+.4f}  "
      f"CI [{_pb['ci95'][0]:+.4f},{_pb['ci95'][1]:+.4f}]  P(<=0) {_pb['p_delta_le_0']:.3f}")


frozen_spec consistent -- safe to proceed:
  model_artifact        : pipe_v2_stable_surface.joblib
  test_set_used         : False   <- must be False
  test_extraction_model : qwen3.5:122b
  frozen_primary        : 20 cols
  prompt sha256[:16]    : f52aae5d939d620e
  CV: AUC 0.8058  paired dAUC +0.0634  CI [+0.0065,+0.1197]  P(<=0) 0.016


## Block 1 — Load the sealed test set

The test-set read is inlined directly in the next cell — the ONLY place in the whole
project that opens `fileDataset/test/`. This is the first and only time these 61
documents are loaded. Surface statistics are computed
identically to the training-set computation (same `surface()` formula) so that the
frozen surface pipeline sees byte-identical features at test time.

In [21]:
# The 61 sealed test transcripts -- the first and only time these are loaded in the
# whole project. This read lives here, in the one notebook allowed to do it.
# test/negative and test/positive mirror the train/ layout. Surface stats use the
# SAME formula as training so the frozen surface pipeline sees the same feature space.
def read_split(folder, label):
    return [{"file": f.name, "label": label,
             "text": f.read_text(encoding="utf-8", errors="replace").strip()}
            for f in sorted(folder.glob("*.txt"))]

_WORD = re.compile(r"\w+", re.UNICODE)
def surface(t):
    nw = max(1, len(t.split()))
    nt = len(set(_WORD.findall(t.lower())))
    ns = max(1, len(re.findall(r"[.!?]+", t)))
    return {"n_char": len(t), "n_word": nw, "n_type": nt, "ttr": nt / nw,
            "n_sent": ns, "mlu": nw / ns, "n_comma": t.count(",")}

rows = read_split(DATA / "test" / "negative", 0) + read_split(DATA / "test" / "positive", 1)
test_docs = pd.DataFrame(rows)
assert len(test_docs) == 61, f"expected 61 test docs, got {len(test_docs)}"
test_docs = pd.concat(
    [test_docs, test_docs.text.apply(lambda t: pd.Series(surface(t)))], axis=1)
y_test = test_docs.label.values.astype(int)

print(f"test docs: {len(test_docs)}")
print(test_docs.label.astype(int).map({0: "negative", 1: "positive"}).value_counts().to_string())
print(f"median words: {test_docs.n_word.median():.0f}  (train median: 69 words)")


test docs: 61
label
negative    42
positive    19
median words: 65  (train median: 69 words)


## Block 2 — Provider

Same `StreamingLocalProvider` as notebooks 08 and 09 — streaming keeps bytes flowing
past the reverse-proxy timeout, `think:false` prevents Qwen3.x from spending the
token budget on internal reasoning, and retries handle transient 5xx / empty-body
failures. Only the model changes relative to notebook 09: MODEL_A (`qwen3.5:122b`)
is used here because the frozen pipeline was trained on MODEL_A extractions.

In [22]:
# Same StreamingLocalProvider as notebooks 08/09 (helpers.py). MODEL_A (qwen3.5:122b)
# because the frozen pipeline was trained on that model's extractions.
provider = make_provider(MODEL)
print("provider ready:", type(provider).__name__, "| model:", provider.text_model)


provider ready: StreamingLocalProvider | model: qwen3.5:122b


## Block 3 — Full test extraction (resumable)

Each response is appended to `v2_test_raw.jsonl` and flushed immediately. If the run
crashes at document 40, re-running this cell with `RUN_LLM = True` skips the first 39
and continues from where it stopped. After a complete run, set `RUN_LLM = False` so
subsequent cells can re-run in analysis-only mode without re-hitting the endpoint.

In [23]:
# resumable JSONL extraction -- inline (same pattern as notebooks 08/09).
def load_raw(path):
    out = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    rec = json.loads(line); out[rec["file"]] = rec
                except Exception:
                    pass
    return out

def seal_last_line(path):
    if path.exists() and path.stat().st_size:
        with path.open("rb+") as fh:
            fh.seek(-1, 2)
            if fh.read(1) != b"\n":
                fh.write(b"\n")

EVIDENCE_FIELDS = ["named_entities", "specific_action_verbs", "generic_verbs",
                   "locative_expressions", "hedge_spans", "deictic_spans",
                   "metacomment_spans", "diminutive_or_affective_forms", "quantity_expressions"]
REQUIRED = EVIDENCE_FIELDS + ["complete_propositions", "regions_referenced",
                              "repeated_content_lemmas", "self_corrections"]

def groundedness(obj, text):
    low, hit, tot = text.lower(), 0, 0
    for f in EVIDENCE_FIELDS:
        for s in obj.get(f, []) or []:
            if isinstance(s, str) and s.strip():
                tot += 1
                hit += s.strip().lower() in low
    return hit / tot if tot else np.nan

def missing_fields(obj):
    return [f for f in REQUIRED if f not in obj]

import time
if RUN_LLM:
    seal_last_line(RAW_TEST)
    done = load_raw(RAW_TEST)
    todo = test_docs[~test_docs.file.isin(done)]
    print(f"already done: {len(done)}   remaining: {len(todo)}")
    t0 = time.time()
    with RAW_TEST.open("a", encoding="utf-8") as fh:
        for i, (_, r) in enumerate(todo.iterrows(), 1):
            rec = {"file": r.file, "label": int(r.label), "model": MODEL, "prompt_sha": PROMPT_SHA}
            try:
                rec["response"] = provider.text_features([r.text], prompt=V2_PROMPT)[0]
                rec["error"] = None
            except Exception as e:
                rec["response"] = None; rec["error"] = f"{type(e).__name__}: {e}"[:300]
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()
            if i % 10 == 0 or i == len(todo):
                el = time.time() - t0
                print(f"  {i}/{len(todo)}  {el/60:.1f} min elapsed, "
                      f"~{el/i*(len(todo)-i)/60:.1f} min left", flush=True)
    raw_test = load_raw(RAW_TEST)
else:
    assert RAW_TEST.exists(), f"{RAW_TEST} not found -- set RUN_LLM = True"
    raw_test = load_raw(RAW_TEST)

n_ok = sum(v["response"] is not None for v in raw_test.values())
print(f"extracted: {n_ok}/{len(raw_test)} succeeded, {len(raw_test) - n_ok} failed")

META_TEST.write_text(json.dumps({
    "model": MODEL, "base_url": BASE_URL, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS,
    "streaming": True, "prompt_version": "v2", "prompt_sha256_16": PROMPT_SHA,
    "prompt_chars": len(V2_PROMPT), "n_documents_sent": len(raw_test), "n_succeeded": n_ok,
    "n_failed": len(raw_test) - n_ok, "n_labelled_for_modelling": 241, "n_unlabelled_extracted": 86,
    "test_set_used": False,   # this is the extraction metadata; the seal goes in frozen_spec.json (Block 8)
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}, indent=2))
print(f"metadata -> {META_TEST}")


already done: 61   remaining: 0
extracted: 61/61 succeeded, 0 failed
metadata -> fileDataset/outputs/v2_test_run_metadata.json


## Block 4 — Quality control

Same three QC checks as notebook 08: failure rate, groundedness (are quoted spans
real?), and missing fields. The frozen pipeline's validity depends on the extraction
being as clean here as it was on the training set (groundedness ≈ 0.98 there). Any
large divergence should be investigated before treating the test-set result as final.

In [24]:
ok_test = {f: v["response"] for f, v in raw_test.items() if v["response"] is not None}

gr = pd.Series({
    f: groundedness(o, test_docs.set_index("file").text[f])
    for f, o in ok_test.items()
})

print(f"usable responses : {len(ok_test)}/{len(raw_test)}  ({len(ok_test)/max(len(raw_test),1):.1%})")
print(f"groundedness     : mean {gr.mean():.3f} | median {gr.median():.3f}")
print(f"  docs below 0.5 : {(gr < 0.5).sum()}")

miss = pd.Series([m for o in ok_test.values() for m in missing_fields(o)]).value_counts()
print(f"\nmissing fields:\n{miss.to_string() if len(miss) else '  none'}")

usable responses : 61/61  (100.0%)
groundedness     : mean 0.990 | median 1.000
  docs below 0.5 : 0

missing fields:
  none


## Block 5 — Build test feature matrix

Identical `build_feature_matrix` call as notebooks 08/09 — same `to_row()` mapping,
same count → per-100-word rate formula. The surface statistics are already in
`test_docs` (computed by `load_test()` using the same `surface()` formula as training).
We assert that every column in `FROZEN_PRIMARY_FEATURES` is present before proceeding.

In [25]:
# raw LLM JSON -> feature matrix -- inline, identical mapping to notebooks 08/09.
LIST_FIELDS = {"n_entities": "named_entities", "n_specific_verb": "specific_action_verbs",
               "n_generic_verb": "generic_verbs", "n_locative": "locative_expressions",
               "n_hedge": "hedge_spans", "n_deictic": "deictic_spans",
               "n_metacomment": "metacomment_spans", "n_diminutive": "diminutive_or_affective_forms",
               "n_quantity": "quantity_expressions"}
INT_FIELDS = {"n_proposition": "complete_propositions", "n_selfcorrect": "self_corrections"}

def to_row(obj):
    r = {}
    for out_, key in LIST_FIELDS.items():
        v = obj.get(key) or []
        r[out_] = len(v) if isinstance(v, list) else 0
    for out_, key in INT_FIELDS.items():
        v = obj.get(key, 0)
        r[out_] = int(v) if isinstance(v, (int, float)) else 0
    reg = obj.get("regions_referenced") or []
    r["n_region"] = len(set(reg)) if isinstance(reg, list) else 0
    rep = obj.get("repeated_content_lemmas") or []
    r["n_repeated_lemma"] = sum(1 for x in rep if isinstance(x, dict) and x.get("lemma"))
    r["repeat_mass"] = sum(int(x.get("count", 0)) for x in rep
                           if isinstance(x, dict) and str(x.get("count", "")).isdigit())
    return r

feat = pd.DataFrame([{"file": f, **to_row(o)} for f, o in ok_test.items()])
dfT = test_docs.merge(feat, on="file", how="inner")
COUNTS_T = [c for c in feat.columns if c != "file"]
for c in COUNTS_T:
    dfT[c + "_r100"] = 100 * dfT[c] / dfT["n_word"].clip(lower=1)
dfT["specific_verb_ratio"] = dfT.n_specific_verb / (dfT.n_specific_verb + dfT.n_generic_verb).clip(lower=1)
dfT["region_breadth"] = dfT.n_region / 3.0
RATES_T  = [c + "_r100" for c in COUNTS_T]
RATIOS_T = ["specific_verb_ratio", "region_breadth"]

missing_cols = [c for c in FROZEN_PRIMARY_FEATURES if c not in dfT.columns]
assert not missing_cols, f"missing columns in test feature matrix: {missing_cols}"

n_feat_missing = dfT[FROZEN_PRIMARY_FEATURES].isna().any(axis=1).sum()
if n_feat_missing > 0:
    print(f"WARNING: {n_feat_missing} docs have NaN features (extraction failed) -- excluded")
    dfT = dfT[dfT[FROZEN_PRIMARY_FEATURES].notna().all(axis=1)].copy()

y_eval = dfT.label.values.astype(int)
print(f"test feature matrix: {dfT.shape[0]} docs  x  {len(COUNTS_T) + len(RATES_T) + len(RATIOS_T)} LLM features")
print(f"all {len(FROZEN_PRIMARY_FEATURES)} frozen_primary columns present: OK")


test feature matrix: 61 docs  x  30 LLM features
all 20 frozen_primary columns present: OK


## Block 6 — Apply frozen pipelines

Both frozen pipelines — the primary (`V2_STABLE + surface`, ElasticNet) and the
surface baseline (`surface` stats only, L2 logistic regression) — are loaded from
disk and applied to the test features. No fitting happens here: `predict_proba()`
on a fitted pipeline is a pure forward pass.

Bootstrap CI is computed on the test set predictions (2000 resamples). With only 61
documents the CI will be wide — that is an honest reflection of the sample size, not
a problem to hide.

In [26]:
art_primary = joblib.load(FROZEN_DIR / "pipe_v2_stable_surface.joblib")
art_surface = joblib.load(FROZEN_DIR / "pipe_surface.joblib")

pipe_primary = art_primary["pipeline"]
pipe_surface = art_surface["pipeline"]

X_primary = dfT[FROZEN_PRIMARY_FEATURES].values
X_surface  = dfT[SURFACE].values

proba_primary = pipe_primary.predict_proba(X_primary)[:, 1]
proba_surface  = pipe_surface.predict_proba(X_surface)[:, 1]

def test_metrics(name, proba):
    auc  = roc_auc_score(y_eval, proba)
    ci   = _boot_ci(y_eval, proba, n=2000, seed=SEED)
    pred = (proba >= 0.5).astype(int)
    return {"model": name,
            "AUC": auc, "CI_low": ci[0], "CI_high": ci[1],
            "balAcc": balanced_accuracy_score(y_eval, pred),
            "macroF1": f1_score(y_eval, pred, average="macro")}

res = [
    test_metrics("Surface stats (frozen baseline)",          proba_surface),
    test_metrics("V2 stable + surface (frozen primary)",     proba_primary),
]
table = pd.DataFrame(res)

print("TEST-SET RESULTS (one-shot, n=61):")
print(table.round(3).to_string(index=False))

TEST-SET RESULTS (one-shot, n=61):
                               model   AUC  CI_low  CI_high  balAcc  macroF1
     Surface stats (frozen baseline) 0.796   0.659    0.909   0.666    0.627
V2 stable + surface (frozen primary) 0.807   0.677    0.916   0.709    0.702


## Block 7 — Paired bootstrap and context

The paired bootstrap on test-set predictions answers the same question as the CV
paired bootstrap in notebook 09 (`dAUC +0.063, CI [+0.004, +0.120], P(<=0)=0.016`):
does V2 add information beyond surface statistics? With n=61 the CI will be wider.
Report both the point estimate and the CI — a wide CI is an honest statement about
the sample size, not a failure of the method.

In [27]:
delta, ci_d, pneg = paired_bootstrap(proba_primary, proba_surface, y_eval, n=2000, seed=SEED)

print("Paired bootstrap: V2 stable+surface  vs  surface alone  (n_boot=2000):")
print(f"  dAUC = {delta:+.4f}   95% CI [{ci_d[0]:+.4f}, {ci_d[1]:+.4f}]   P(dAUC<=0) = {pneg:.3f}")
print()

cv_auc = spec["frozen_primary_cv_auc"]
cv_pb  = spec["paired_bootstrap"]["frozen_primary_vs_surface"]
auc_test_primary = table.loc[table.model.str.startswith("V2"), "AUC"].iloc[0]
auc_test_surface = table.loc[table.model.str.startswith("Surface"), "AUC"].iloc[0]

print("Context — training CV vs test:")
print(f"  CV  primary AUC : {cv_auc:.4f}")
print(f"  test primary AUC: {auc_test_primary:.4f}")
print(f"  CV  paired dAUC : {cv_pb['delta_auc']:+.4f}  CI {cv_pb['ci95']}  P(<=0) {cv_pb['p_delta_le_0']:.3f}")
print(f"  test paired dAUC: {delta:+.4f}  CI [{ci_d[0]:+.4f},{ci_d[1]:+.4f}]  P(<=0) {pneg:.3f}")

Paired bootstrap: V2 stable+surface  vs  surface alone  (n_boot=2000):
  dAUC = +0.0109   95% CI [-0.1034, +0.1240]   P(dAUC<=0) = 0.421

Context — training CV vs test:
  CV  primary AUC : 0.8058
  test primary AUC: 0.8070
  CV  paired dAUC : +0.0634  CI [0.006470110667049964, 0.11973858587914607]  P(<=0) 0.016
  test paired dAUC: +0.0109  CI [-0.1034,+0.1240]  P(<=0) 0.421


## Block 8 — Seal the test set

Write the test results and `test_set_used: true` into `frozen_spec.json`. After this
cell runs, any future attempt to re-run this notebook will fail at Block 0 because
`assert_consistent()` checks `test_set_used is False` and raises if it is not.
This is the only automatic guard against accidentally re-evaluating after seeing
the result.

In [28]:
spec_path    = FROZEN_DIR / "frozen_spec.json"
spec_updated = json.loads(spec_path.read_text())
assert not spec_updated["test_set_used"], (
    "test_set_used is already True — test set has been evaluated, do not re-run"
)

_pb_test = {
    "contrast":     "V2_STABLE+surface (frozen primary) vs surface (frozen baseline)",
    "n_resamples":  2000,
    "delta_auc":    float(delta),
    "ci95":         [float(ci_d[0]), float(ci_d[1])],
    "p_delta_le_0": float(pneg),
}
_row_primary = table.loc[table.model.str.startswith("V2")].iloc[0]
_row_surface = table.loc[table.model.str.startswith("Surface")].iloc[0]

spec_updated["test_set_used"]     = True
spec_updated["test_set_used_utc"] = (
    datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
)
spec_updated["test_results"] = {
    "n_test":            int(len(y_eval)),
    "model_artifact":    spec["model_artifact"],
    "primary_auc":       float(_row_primary.AUC),
    "primary_ci95":      [float(_row_primary.CI_low), float(_row_primary.CI_high)],
    "primary_bal_acc":   float(_row_primary.balAcc),
    "primary_macro_f1":  float(_row_primary.macroF1),
    "surface_auc":       float(_row_surface.AUC),
    "paired_bootstrap":  _pb_test,
}
spec_path.write_text(json.dumps(spec_updated, indent=2, default=str))

print("frozen_spec.json updated: test_set_used = True")
print(f"  primary AUC  : {spec_updated['test_results']['primary_auc']:.4f}")
print(f"  surface AUC  : {spec_updated['test_results']['surface_auc']:.4f}")
print(f"  paired dAUC  : {_pb_test['delta_auc']:+.4f}  CI {_pb_test['ci95']}  P(<=0) {_pb_test['p_delta_le_0']:.3f}")

frozen_spec.json updated: test_set_used = True
  primary AUC  : 0.8070
  surface AUC  : 0.7957
  paired dAUC  : +0.0109  CI [-0.10337936046511631, 0.12403998865570057]  P(<=0) 0.421


## Block 9 — Verdict

Three outcomes are all publishable. The key check is whether the test-set AUC is
consistent with the CV estimate (expected: within sampling noise for n=61) and
whether the V2 improvement over surface holds out-of-sample.

In [29]:
tr = spec_updated["test_results"]
print("=" * 70)
print("FINAL TEST-SET RESULT  (n=61, one-shot)")
print("=" * 70)
print(f"V2 stable + surface (frozen primary)  AUC {tr['primary_auc']:.4f}  "
      f"95% CI [{tr['primary_ci95'][0]:.4f}, {tr['primary_ci95'][1]:.4f}]")
print(f"  balAcc {tr['primary_bal_acc']:.3f}  macroF1 {tr['primary_macro_f1']:.3f}")
print(f"Surface baseline (frozen)             AUC {tr['surface_auc']:.4f}")
print()

_pb = tr["paired_bootstrap"]
print(f"Paired bootstrap dAUC {_pb['delta_auc']:+.4f}  "
      f"CI [{_pb['ci95'][0]:+.4f},{_pb['ci95'][1]:+.4f}]  P(<=0) {_pb['p_delta_le_0']:.3f}")
print()

if _pb["ci95"][0] > 0:
    print("-> V2 features ADD to surface on held-out test data (CI entirely above zero).")
elif _pb["ci95"][1] < 0:
    print("-> V2 features HURT surface on held-out test data (CI entirely below zero).")
else:
    print("-> Test-set CI crosses zero — directionally consistent with CV but underpowered at n=61.")
    print("   Report point estimate and CI; do not claim significance from the test set alone.")

print()
cv_auc = spec["frozen_primary_cv_auc"]
gap = tr["primary_auc"] - cv_auc
if abs(gap) < 0.05:
    print(f"CV AUC {cv_auc:.4f} vs test AUC {tr['primary_auc']:.4f} — gap {gap:+.4f} within ±0.05, no overfit signal.")
elif gap < -0.05:
    print(f"CV AUC {cv_auc:.4f} vs test AUC {tr['primary_auc']:.4f} — gap {gap:+.4f} > 0.05: "
          f"possible overfit or class-imbalance effect; report alongside the CI width.")
else:
    print(f"CV AUC {cv_auc:.4f} vs test AUC {tr['primary_auc']:.4f} — gap {gap:+.4f}: "
          f"test slightly above CV, consistent with random variation on n=61.")

print()
print("This notebook is now sealed. frozen_spec.json carries test_set_used = True.")

FINAL TEST-SET RESULT  (n=61, one-shot)
V2 stable + surface (frozen primary)  AUC 0.8070  95% CI [0.6768, 0.9159]
  balAcc 0.709  macroF1 0.702
Surface baseline (frozen)             AUC 0.7957

Paired bootstrap dAUC +0.0109  CI [-0.1034,+0.1240]  P(<=0) 0.421

-> Test-set CI crosses zero — directionally consistent with CV but underpowered at n=61.
   Report point estimate and CI; do not claim significance from the test set alone.

CV AUC 0.8058 vs test AUC 0.8070 — gap +0.0012 within ±0.05, no overfit signal.

This notebook is now sealed. frozen_spec.json carries test_set_used = True.
